# NGS Amplicon Analysis Pipeline for CRISPR Genome Editing Assessment

**Author:** Innocent Byiringiro | University of Maryland, College Park | [ibyiring@umd.edu](mailto:ibyiring@umd.edu)

## Overview

This notebook implements an end-to-end pipeline for analyzing CRISPR-Cas genome editing outcomes from Illumina MiSeq amplicon sequencing data. The pipeline:

1. **Merges** paired-end reads using FLASH-based overlap detection
2. **Demultiplexes** merged reads by barcode pairs
3. **Quantifies** editing outcomes (indel frequencies, allele distributions) using [CRISPResso2](https://github.com/pinellolab/CRISPResso2)
4. **Exports** results for downstream visualization

## How to Use

1. Upload your paired-end `.fastq.gz` files to the Colab environment
2. Prepare a barcode CSV file mapping sample names to barcode sequences
3. Run cells sequentially, updating file paths and parameters as needed
4. Download the zipped CRISPResso2 output for further analysis

## Requirements

- Google Colab (recommended) or local Jupyter with CRISPResso2 installed
- Paired-end Illumina MiSeq FASTQ files
- Barcode reference CSV with columns: `Sample`, `Barcode_L`, `Barcode_R`


---
## 1. Environment Setup

Install dependencies and import required packages. This cell clones [flashpy](https://github.com/ponnhide/flashpy) for read merging and installs BioPython.

In [ ]:
# Install and import dependencies
import subprocess
import os
import sys
import gzip
import collections
import numpy as np
from tqdm import tqdm
import shutil
import pandas as pd

!pip install -q Biopython
from Bio import SeqIO

# Clone and build flashpy for read merging
!git clone -q https://github.com/ponnhide/flashpy.git
%cd flashpy
subprocess.run('python setup.py build_ext --inplace', shell=True, capture_output=True)
import _flash as fl

# R integration for downstream plotting (optional)
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

# Uncomment below to mount Google Drive if your data is stored there
# from google.colab import drive
# drive.mount('/content/drive')

print('Environment setup complete.')

---
## 2. Read Merging Functions

Define functions for merging paired-end reads based on overlap detection.

The `flash()` function reads paired FASTQ files and merges overlapping read pairs using either a Cython-optimized or pure Python implementation. Key parameters:
- `min_overlap`: Minimum overlap length to consider (default: 10 bp)
- `max_overlap`: Maximum overlap length (default: 300 bp)
- `min_identity`: Minimum sequence identity in overlap region (default: 0.1)

Quality scores are used to resolve mismatches: the base with the higher quality score is kept in the merged sequence.

In [ ]:
#@title 2. Merging reads

#@markdown Please execute this cell by pressing the *Play* button on
#@markdown the left. You can show the code and adjust merge/flash parameters
#@markdown for example you can change the min and max overlaps.

hamming = lambda x,y: sum(tuple(map(lambda a,b: 1 if a==b else 0, x, y))) / len(x)
convert_ascii = lambda x: [ord(asc) - 33 for asc in x]
def read_fastq(fastq_name):
    """Read fastq file
    """
    seq_dict = collections.defaultdict(dict)
    if fastq_name.split(".")[-1] == "gz":
        f = gzip.open(fastq_name.replace("'","").replace("\\",""), mode="rt", encoding='utf-8')
    else:
        f = open(fastq_name.replace("'","").replace("\\",""))
    n = 0
    for line in f:
        if line[0] == "@" and n % 4 == 0:
            key = line[1:].rstrip()
            key = key.split(" ")[0]
            key = key.replace(":","_")
            seq_dict[key]["key"] = line[1:].rstrip()
        elif n%4 == 1:
            seq_dict[key]["seq"] = line.rstrip()
        elif n%4 == 2:
            seq_dict[key]["option"] = line.rstrip()
        elif n%4 == 3:
            seq_dict[key]["quality"] = [ord(asc) - 33 for asc in line.rstrip()]
        n += 1
    f.close()
    return seq_dict

def merge(seq1, seq2, score1, score2, min_overlap=10, max_overlap=300, allow_outies=True, min_identity=0.1, max_identity=1.0, cython=True):
    seq1 = seq1.upper()
    seq2 = seq2.upper()
    reverse = 0
    if len(seq1) >= len(seq2):
        seq2 = seq2.translate(str.maketrans("ATGCRYKMSWBDHV","TACGYRMKWSVHDB"))[::-1]
        score2 = score2[::-1]
    else:
        reverse = 1
        seq1 = seq1.translate(str.maketrans("ATGCRYKMSWBDHV","TACGYRMKWSVHDB"))[::-1]
        seq2, seq1 = seq1, seq2

    if cython == False:
        current_overlap  = 0
        current_score    = None
        current_identity = min_identity
        current_slide     = 0
        for i in range(len(seq1)+len(seq2)):
            slide = i
            if i < len(seq2) and allow_outies == True:
                overlap_length = i
                subseq1   = seq1[:overlap_length]
                subseq2   = seq2[-1*overlap_length:]
                subscore1 = score1[:overlap_length]
                subscore2 = score2[-1*overlap_length:]
            elif i < len(seq1):
                overlap_length = len(seq2)
                subseq1   = seq1[i-len(seq2):i]
                subseq2   = seq2
                subscore1 = score1[i-len(seq2):i]
                subscore2 = score2
            else:
                overlap_length = len(seq1) + len(seq2) - i
                subseq1   = seq1[-1*overlap_length:]
                subseq2   = seq2[:overlap_length:]
                subscore1 = score1[-1*overlap_length:]
                subscore2 = score2[:overlap_length]

            if min_overlap <= overlap_length <= max_overlap:
                identity = hamming(subseq1, subseq2)
                if identity > current_identity or (identity == current_identity and current_overlap == 0):
                    current_slide     = slide
                    current_identity  = identity
                    current_overlap   = overlap_length
                    current_subseq1   = subseq1
                    current_subseq2   = subseq2
                    current_subscore1 = subscore1
                    current_subscore2 = subscore2
                    #for n1, n2, s1, s2 in zip(subseq1, subseq2, subscore1, subscore2):

                elif identity == current_identity:
                    n = 0
                    cscore_avg = 0
                    for cn1, cn2, cs1, cs2 in zip(current_subseq1, current_subseq2, current_subscore1, current_subscore2):
                        if cn1 != cn2:
                            cscore_avg += max(cs1, cs2)
                            n += 1
                    csocre_avg = cscore_avg / n

                    n = 0
                    score_avg = 0
                    for n1, n2, s1, s2 in zip(subseq1, subseq2, subscore1, subscore2):
                        if n1 != n2:
                            score_sum += max(s1, s2)
                            n += 1
                    score_avg = score_avg / n
                    if score_avg > cscore_avg:
                        current_slide     = slide
                        current_identity  = identity
                        current_overlap   = overlap_length
                        current_subseq1   = subseq1
                        current_subseq2   = subseq2
                        current_subscore1 = subscore1
                        current_subscore2 = subscore2
                else:
                    pass

                if identity >= max_identity:
                    break

        if current_identity < min_identity:
            return False
        else:
            overlap_seq   = ""
            overlap_score = []
            for cn1, cn2, cs1, cs2 in zip(current_subseq1, current_subseq2, current_subscore1, current_subscore2):
                if cs1 > cs2:
                    overlap_seq += cn1
                    overlap_score.append(cs1)
                else:
                    overlap_seq += cn2
                    overlap_score.append(cs2)

            if current_slide < len(seq2) and allow_outies == True:
                left_seq    = seq2[:-1*overlap_length]
                right_seq   = seq1[overlap_length:]
                left_score  = score2[:-1*overlap_length]
                right_score = score1[overlap_length:]

            else:
                left_seq    = seq1[:-1*overlap_length]
                right_seq   = seq2[overlap_length:]
                left_score  = score1[:-1*overlap_length]
                right_score = score2[overlap_length:]
            merged_seq   = left_seq + overlap_seq + right_seq
            merged_score = left_score + overlap_score + right_score
    else:
        merged_seq, merged_score, current_slide, current_overlap, current_identity = fl.merge(seq1.encode('utf-8'), seq2.encode('utf-8'), score1, score2, min_overlap, max_overlap, allow_outies, min_identity, max_identity)
    return merged_seq, merged_score, current_slide, current_overlap, current_identity

def flash(read1, read2, min_overlap=10, max_overlap=300, allow_outies=False, min_identity=0.1, max_identity=1.0, show_progress=True, key_check=True):
    r1_dict = read_fastq(read1)
    r2_dict = read_fastq(read2)

    dist_dict = collections.defaultdict(lambda:[0, 0])
    merged_dict = collections.defaultdict(dict)
    if show_progress == True:
        if key_check == True:
            keys = [key for key in r1_dict.keys() if key in r2_dict]
            keys = tqdm(keys)
        else:
            keys = tqdm(r1_dict.keys(), total=len(r1_dict))
    else:
        if key_check == True:
            keys = [key for key in r1_dict.keys() if key in r2_dict]
        else:
            keys = r1_dict.keys()

    for key in keys:
        seq1   = r1_dict[key]["seq"]
        seq2   = r2_dict[key]["seq"]
        score1 = r1_dict[key]["quality"]
        score2 = r2_dict[key]["quality"]
        result = merge(seq1, seq2, score1, score2, min_overlap, max_overlap, allow_outies, min_identity, max_identity)
        if result != False:
            merged_dict[key]["seq"]      = "".join(map(chr, result[0]))
            merged_dict[key]["quality"]  = result[1]
            merged_dict[key]["r1_key"]   = r1_dict[key]["key"]
            merged_dict[key]["r2_key"]   = r2_dict[key]["key"]
            merged_dict[key]["identity"] = result[4]
            if result[2] == 1:
                dist_dict["outie", result[3]][0] += 1
                dist_dict["outie", result[3]][1] += result[4]
            else:
                dist_dict["innie", result[3]][0] += 1
                dist_dict["innie", result[3]][1] += result[4]
    for key in dist_dict:
        dist_dict[key][1] = dist_dict[key][1] / dist_dict[key][0]

    return merged_dict, dist_dict

---
## 3. Merge Paired-End Reads

Specify your paired-end FASTQ files (R1 and R2) and merge them.

**Update the file paths below** to point to your uploaded FASTQ files. Both `.fastq` and `.fastq.gz` formats are supported.

In [ ]:
# ============================================================
# UPDATE THESE PATHS to point to your FASTQ files
# ============================================================
read_R1 = '/content/YOUR_SAMPLE_R1_001.fastq.gz'  #@param {type:"string"}
read_R2 = '/content/YOUR_SAMPLE_R2_001.fastq.gz'  #@param {type:"string"}
output_file_name = 'YOUR_SAMPLE_merged'            #@param {type:"string"}

# Merge paired-end reads
merged_dict, dist_dict = flash(read_R1, read_R2)

# Print overlap statistics
print(f'Total merged reads: {len(merged_dict)}')
print(f'\nOverlap distribution (top 10):')
sorted_dist = sorted(dist_dict.items(), key=lambda x: x[1][0], reverse=True)[:10]
for key, val in sorted_dist:
    print(f'  {key[0]} overlap={key[1]}bp: count={val[0]}, avg_identity={val[1]:.3f}')

# Write merged reads to FASTQ
n = 0
with open(output_file_name, 'w') as f:
    for key in merged_dict:
        f.write(f"@{key}\n{merged_dict[key]['seq']}\n")
        n += 1
print(f'\nWrote {n} merged reads to {output_file_name}')

---
## 4. Demultiplex by Barcodes

Split merged reads into sample-specific FASTQ files based on barcode pairs.

The function checks both forward and reverse-complement orientations of the barcode pairs, handling reads that may be in either direction. Each sample's reads are written to a separate directory.

In [ ]:
#@title 4. Command for splitting based on the barcodes

#@markdown Please execute this cell by pressing the *Play* button on
#@markdown the left.

def reverse_complement(dna):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join([complement[base] for base in dna[::-1]])

def split_fastq(df, fastq, output_directory):
    for Index, row in df.iterrows():
        sample = row['Sample']
        left_code = row['Barcode_L'].upper()
        right_code_raw = row['Barcode_R'].upper()
        right_code = reverse_complement(right_code_raw)
        left_length = len(left_code)
        right_length = len(right_code)
        left_code_reverse = reverse_complement(left_code)
        right_code_reverse = reverse_complement(left_code)

        # Create the output directory
        output_dir = os.path.join(output_directory, sample)
        os.makedirs(output_dir, exist_ok=True)

        # Define the output Fastq file path
        output_fastq_path = os.path.join(output_dir, f'{sample}.extendedFrags.fastq')

        # Open input and output Fastq files using context managers
        with open(fastq, 'r') as input_fastq, open(output_fastq_path, 'w') as output_fastq:
            flag = 0
            for line in input_fastq:
                inf = line.rstrip()
                flag += 1
                if flag == 1:
                    name = inf
                elif flag == 2:
                    seq = inf
                elif flag == 4:
                    quality = inf
                    flag = 0
                    if seq[:left_length] == left_code and seq[-right_length:] == right_code:
                        seq1 = seq[left_length:-right_length]
                        quality1 = quality[left_length:-right_length]
                        print(name, seq1, '+', quality1, sep='\n', file=output_fastq)
                    elif seq[:right_length] == right_code_raw and seq[-left_length:] == left_code_reverse:
                        seq1 = seq[right_length:-left_length]
                        quality1 = quality[right_length:-left_length]
                        print(name, seq1, '+', quality1, sep='\n', file=output_fastq)

### 4b. Run Demultiplexing

Provide:
- **reference_df**: Path to your barcode CSV file (columns: `Sample`, `Barcode_L`, `Barcode_R`)
- **merged_reads**: Path to the merged FASTQ file from Step 3
- **output**: Directory name for the split output files

In [ ]:
# ============================================================
# UPDATE THESE PATHS for your experiment
# ============================================================
reference_df = pd.read_csv('/content/YOUR_BARCODE_TABLE.csv')  #@param {type:"string"}
merged_reads = '/content/flashpy/YOUR_SAMPLE_merged'           #@param {type:"string"}
output = 'YOUR_SAMPLE_Split'                                    #@param {type:"string"}

# Run demultiplexing
split_fastq(df=reference_df, fastq=merged_reads, output_directory=output)
print(f'Demultiplexing complete. Output in: {output}/')

---
## 5. Install CRISPResso2

[CRISPResso2](https://github.com/pinellolab/CRISPResso2) is a tool for quantifying genome editing outcomes from deep sequencing data. This cell installs it along with FLASH for additional read merging support.

In [ ]:
#@title 6. Download CRISPResso2 from github

#@markdown Please execute this cell by pressing the *Play* button on
#@markdown the left.

# 1. Installation of necessary tools and CRISPResso2

!apt-get install -y git
!git clone https://github.com/pinellolab/CRISPResso2.git
%cd CRISPResso2
!pip install .

!wget http://ccb.jhu.edu/software/FLASH/FLASH-1.2.11-Linux-x86_64.tar.gz
!tar -zxvf FLASH-1.2.11-Linux-x86_64.tar.gz
!cp FLASH-1.2.11-Linux-x86_64/flash /usr/local/bin/

---
## 6. Prepare CRISPResso2 Batch File

Generate a batch settings file from the demultiplexed samples. This file maps each sample to its FASTQ path, amplicon sequence, and guide RNA.

**Update the parameters below:**
- `project_directory`: Path to your demultiplexed output folder
- `amplicon_seq`: The full amplicon reference sequence
- `guide_seq`: The 20-nt guide RNA sequence (without PAM)

In [ ]:
# ============================================================
# UPDATE THESE PARAMETERS for your experiment
# ============================================================
project_directory = '/content/flashpy/YOUR_SAMPLE_Split'  #@param {type:"string"}
amplicon_seq = 'YOUR_AMPLICON_SEQUENCE_HERE'              #@param {type:"string"}
guide_seq = 'YOUR_GUIDE_RNA_20NT'                         #@param {type:"string"}
batch_settings_file = '/content/flashpy/YOUR_SAMPLE_batch' #@param {type:"string"}

batch_settings = pd.DataFrame(columns=['name', 'fastq_r1', 'amplicon_seq', 'guide_seq', 'editing_type'])

ignore_dirs = ['.ipynb_checkpoints']

for sample_name in os.listdir(project_directory):
    if sample_name in ignore_dirs:
        continue

    sample_path = os.path.join(project_directory, sample_name)
    if os.path.isdir(sample_path):
        fastq_files = [f for f in os.listdir(sample_path) if f.endswith('.fastq')]
        if not fastq_files:
            print(f'No .fastq files found in {sample_path}. Skipping...')
            continue
        fastq_r1 = os.path.join(sample_path, fastq_files[0])

        new_row = pd.DataFrame({
            'name': [sample_name],
            'fastq_r1': [fastq_r1],
            'amplicon_seq': [amplicon_seq],
            'guide_seq': [guide_seq],
            'editing_type': ['genome']
        })
        batch_settings = pd.concat([batch_settings, new_row], ignore_index=True)

batch_settings.to_csv(batch_settings_file, sep='\t', index=False)
print(f'Batch file created with {len(batch_settings)} samples: {batch_settings_file}')
batch_settings.head()

---
## 7. Run CRISPResso2 Batch Analysis

Execute CRISPResso2 in batch mode across all demultiplexed samples.

**Parameters:**
- `window_center`: Position of the quantification window center relative to the cut site (default: -10)
- `window_size`: Size of the quantification window in bp (default: 20)

In [ ]:
# ============================================================
# CRISPResso2 analysis parameters
# ============================================================
batch_file = batch_settings_file  # From Step 6
window_center = -10  #@param {type:"integer"}
window_size = 20     #@param {type:"integer"}

# Run CRISPResso2 batch analysis
!CRISPRessoBatch --batch_settings {batch_file} --amplicon_seq {amplicon_seq} -g {guide_seq} -wc {window_center} -w {window_size}

---
## 8. Export Results

Zip and download the CRISPResso2 output directory for downstream analysis and visualization.

In [ ]:
# Zip the CRISPResso2 output
import glob

# Find the CRISPRessoBatch output directory
batch_output = glob.glob('CRISPRessoBatch_on_*')
if batch_output:
    zip_name = batch_output[0]
    !zip -r {zip_name}.zip {zip_name}
    print(f'Zipped: {zip_name}.zip')

    # Download (Colab only)
    from google.colab import files
    files.download(f'{zip_name}.zip')
else:
    print('No CRISPRessoBatch output found. Check the previous step for errors.')

---
## 9. Downstream Visualization

CRISPResso2 generates multiple output formats for visualization:

1. **Built-in figures**: CRISPResso2 produces publication-ready plots (HTML and PNG) for each sample
2. **Python visualization**: Use matplotlib/seaborn with the summary text files
3. **R visualization**: Use ggplot2 with the summary text files (recommended for publication figures)

The batch output directory contains per-sample folders with:
- `CRISPResso_quantification_of_editing_frequency.txt` - Overall editing rates
- `Alleles_frequency_table.zip` - Detailed allele frequencies
- `CRISPResso_mapping_statistics.txt` - Read mapping quality metrics

---

**Contact:** Innocent Byiringiro ([ibyiring@umd.edu](mailto:ibyiring@umd.edu)) | Qi Lab, University of Maryland